# 01 — PBDB Harvester
### Project IceWave | Pleistocene Megafauna | WA · OR · NV

Pulls Pleistocene mammal occurrences from the Paleobiology Database for three states.
Targets: Columbian mammoth, woolly mammoth, mastodon, ground sloth, Pleistocene horse, camel.


In [1]:
import requests
import pandas as pd
import geopandas as gpd
import json
from pathlib import Path
from shapely.geometry import Point

Path('../data/pbdb').mkdir(parents=True, exist_ok=True)

# ── Target taxa ────────────────────────────────────────────────────────────
TAXA = [
    'Mammuthus',        # mammoths
    'Mammut',           # mastodons
    'Megalonyx',        # ground sloths
    'Nothrotheriops',   # Shasta ground sloth
    'Equus',            # Pleistocene horses
    'Camelops',         # Pleistocene camels
    'Paramylodon',      # Harlan's ground sloth
]

# ── Bounding boxes per state ───────────────────────────────────────────────
STATES = {
    'WA': {'lngmin': -124.8, 'lngmax': -116.9, 'latmin':  45.5, 'latmax':  49.1},
    'OR': {'lngmin': -124.6, 'lngmax': -116.5, 'latmin':  41.9, 'latmax':  46.3},
    'NV': {'lngmin': -120.0, 'lngmax': -114.0, 'latmin':  35.0, 'latmax':  42.1},
}

BASE_URL = 'https://paleobiodb.org/data1.2/occs/list.json'

all_records = []

for taxon in TAXA:
    for state, bbox in STATES.items():
        params = {
            'base_name': taxon,
            'interval':  'Pleistocene',
            'lngmin':    bbox['lngmin'],
            'lngmax':    bbox['lngmax'],
            'latmin':    bbox['latmin'],
            'latmax':    bbox['latmax'],
            'show':      'coords,loc,time,strat',
            'limit':     'all',
        }
        r = requests.get(BASE_URL, params=params, timeout=30)
        if r.status_code != 200:
            print(f'  ERROR {r.status_code}: {taxon} / {state}')
            continue
        data = r.json()
        records = data.get('records', [])
        for rec in records:
            rec['_taxon_query'] = taxon
            rec['_state']       = state
        all_records.extend(records)
        print(f'  {taxon:20s} {state}: {len(records):4d} occurrences')

print(f'\nTotal raw records: {len(all_records)}')


ReadTimeout: HTTPSConnectionPool(host='paleobiodb.org', port=443): Read timed out. (read timeout=30)

In [ ]:
# ── Parse into DataFrame ──────────────────────────────────────────────────
rows = []
for r in all_records:
    try:
        lat = float(r.get('lat', 0))
        lng = float(r.get('lng', 0))
    except:
        continue
    if lat == 0 and lng == 0:
        continue
    rows.append({
        'occurrence_no': r.get('oid', ''),
        'taxon_name':    r.get('tna', r.get('genus_name', '')),
        'taxon_query':   r.get('_taxon_query', ''),
        'state':         r.get('_state', ''),
        'latitude':      lat,
        'longitude':     lng,
        'formation':     r.get('formation', ''),
        'stratgroup':    r.get('stratgroup', ''),
        'member':        r.get('member', ''),
        'early_interval':r.get('early_interval', ''),
        'late_interval': r.get('late_interval', ''),
        'collection_no': r.get('cid', ''),
        'reference_no':  r.get('rid', ''),
    })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=['occurrence_no'])
print(f'Unique occurrences: {len(df)}')
print(f'\nBy taxon:')
print(df['taxon_query'].value_counts().to_string())
print(f'\nBy state:')
print(df['state'].value_counts().to_string())


In [ ]:
# ── Save CSV ──────────────────────────────────────────────────────────────
csv_path = '../data/pbdb/icewave_occurrences.csv'
df.to_csv(csv_path, index=False)
print(f'Saved CSV: {csv_path} ({len(df)} rows)')


In [ ]:
# ── Build GeoJSON ─────────────────────────────────────────────────────────
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(row.longitude, row.latitude) for _, row in df.iterrows()],
    crs='EPSG:4326'
)

geojson_path = '../data/pbdb/icewave_occurrences.geojson'
gdf.to_file(geojson_path, driver='GeoJSON')
print(f'Saved GeoJSON: {geojson_path}')
print(f'\nSample records:')
print(gdf[['taxon_name','state','latitude','longitude','formation']].head(10).to_string())


In [ ]:
# ── Summary map ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('#1b4332')

colors = {
    'Mammuthus':     '#f0c040',
    'Mammut':        '#52b788',
    'Megalonyx':     '#74c0fc',
    'Nothrotheriops':'#ff8fab',
    'Equus':         '#ffa94d',
    'Camelops':      '#da77f2',
    'Paramylodon':   '#63e6be',
}

states_info = [
    ('WA', -124.8, -116.9, 45.5, 49.1),
    ('OR', -124.6, -116.5, 41.9, 46.3),
    ('NV', -120.0, -114.0, 35.0, 42.1),
]

for ax, (state, lngmin, lngmax, latmin, latmax) in zip(axes, states_info):
    ax.set_facecolor('#0a1a0f')
    sub = df[df['state'] == state]
    for taxon, grp in sub.groupby('taxon_query'):
        ax.scatter(grp['longitude'], grp['latitude'],
                   c=colors.get(taxon, '#ffffff'), s=40, alpha=0.8,
                   label=taxon, edgecolors='none', zorder=3)
    ax.set_xlim(lngmin, lngmax)
    ax.set_ylim(latmin, latmax)
    ax.set_title(f'{state} — {len(sub)} occurrences',
                 color='#f0e8d0', fontsize=13, fontweight='bold')
    ax.tick_params(colors='#8b9bb4')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d6a4f')
    ax.grid(True, color='#2d6a4f', linewidth=0.4, alpha=0.5)

patches = [mpatches.Patch(color=c, label=t) for t, c in colors.items()]
fig.legend(handles=patches, loc='lower center', ncol=4,
           facecolor='#1b4332', edgecolor='#c9a84c',
           labelcolor='#f0e8d0', fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Project IceWave — PBDB Occurrences by State & Taxon',
             color='#c9a84c', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/pbdb/occurrence_map.png', dpi=150, bbox_inches='tight',
            facecolor='#1b4332')
plt.show()
print('Map saved.')


## Results Summary
- CSV: `data/pbdb/icewave_occurrences.csv`
- GeoJSON: `data/pbdb/icewave_occurrences.geojson`
- Map: `data/pbdb/occurrence_map.png`

**Next:** Run `02_terrain_analysis.ipynb` to download USGS DEM tiles for the study area.
